Getting started in Azure Databricks

In [0]:
csv_path = "/Workspace/Users/azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_bookings.csv"
json_path = "/Workspace/Users/azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_master.json"

A. Read and inspect the data

In [0]:
#1
bookings_df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)
#2
cinemas_df = spark.read.json(
    json_path
)
display(bookings_df)
display(cinemas_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


cinema_id,cinema_name,facilities,location,screen_count
C001,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,Coastal Cinema,List(Parking),"List(Chennai, South)",3
C005,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4


In [0]:
#3
bookings_df.printSchema()
cinemas_df.printSchema()

root
 |-- booking_id: string (nullable = true)
 |-- cinema_id: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- show_date: date (nullable = true)
 |-- tickets: integer (nullable = true)
 |-- price_per_ticket: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_method: string (nullable = true)

root
 |-- cinema_id: string (nullable = true)
 |-- cinema_name: string (nullable = true)
 |-- facilities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zone: string (nullable = true)
 |-- screen_count: long (nullable = true)



In [0]:
#4
print("First 10 bookings")
display(bookings_df.limit(10))
print("Total number of csv rows:",bookings_df.count())

First 10 bookings


booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


Total number of csv rows: 32


In [0]:
#5
bookings_df.select(
    'booking_id',
    'movie_title',
    'tickets',
    'booking_status'
).show()

+----------+-----------------+-------+--------------+
|booking_id|      movie_title|tickets|booking_status|
+----------+-----------------+-------+--------------+
|     B1001|Starlight Express|      2|     Confirmed|
|     B1002|  The Last Island|      3|     Confirmed|
|     B1003|         Code Red|      1|     Cancelled|
|     B1004|  River of Dreams|      4|     Confirmed|
|     B1005|Starlight Express|      2|     Confirmed|
|     B1006|  The Last Island|      5|     Confirmed|
|     B1007|         Code Red|      3|     Confirmed|
|     B1008|  River of Dreams|      2|      Refunded|
|     B1009|Starlight Express|      4|     Confirmed|
|     B1010|  The Last Island|      1|     Confirmed|
|     B1011|         Code Red|      2|     Confirmed|
|     B1012|  River of Dreams|      3|     Cancelled|
|     B1013|  The Last Island|      5|     Confirmed|
|     B1014|Starlight Express|      2|     Confirmed|
|     B1015|  River of Dreams|      1|     Confirmed|
|     B1016|         Code Re

B. Filter, clean and transform

In [0]:
#6
from pyspark.sql.functions import col

bookings_df.filter(
    col('booking_status') == 'Confirmed'
).show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|
|     B1004|     C004|  River of Dreams|  2026-10-02|2026-10-05|      4|             180|           0|     Confirmed|          Cash|
|     B1005|     C005|Starlight Express|  2026-10-03|2026-10-05|      2|             320|          15|     Confirmed|          Card|
|     B1006|     C006|  The Last Island|  2026-10-03|2026-10-04|     

In [0]:
#7
bookings_df.filter(
    (col('tickets') > 3) & 
    (col('price_per_ticket') > 200)
).show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|     B1006|     C006|  The Last Island|  2026-10-03|2026-10-04|      5|             210|        NULL|     Confirmed|           UPI|
|     B1009|     C003|Starlight Express|  2026-10-05|2026-10-07|      4|             220|           5|     Confirmed|           UPI|
|     B1013|     C001|  The Last Island|  2026-10-07|2026-10-09|      5|             250|          10|     Confirmed|          Card|
|     B1017|     C005|  The Last Island|  2026-10-09|2026-10-11|      4|             320|          15|      Refunded|          Card|
|     B1023|     C005|  River of Dreams|  2026-10-12|2026-10-14|     

In [0]:
#8
from pyspark.sql.functions import when, sum, col
bookings_df.select(
    sum(when(col('discount_pct').isNull(),1).otherwise(0)).alias('null_discount'),
    sum(when(col('payment_method').isNull(),1).otherwise(0)).alias('null_payment_method')   
).show()

+-------------+-------------------+
|null_discount|null_payment_method|
+-------------+-------------------+
|            5|                  2|
+-------------+-------------------+



In [0]:
#9
cleaned_booking_df = bookings_df.fillna(
    {
        'discount_pct':0,
        'payment_method':'Unknown'
    }
)
#Confirming the null replacement
cleaned_booking_df.select(
    sum(when(col('discount_pct').isNull(),1).otherwise(0)).alias('null_discount'),
    sum(when(col('payment_method').isNull(),1).otherwise(0)).alias('null_payment_method')   
).show()


+-------------+-------------------+
|null_discount|null_payment_method|
+-------------+-------------------+
|            0|                  0|
+-------------+-------------------+



In [0]:
#10
cleaned_booking_df = cleaned_booking_df.withColumn(
    'gross_amount', 
    col('tickets') * col('price_per_ticket')
)
display(cleaned_booking_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180


In [0]:
#11
cleaned_booking_df = cleaned_booking_df.withColumn(
    'discount_amount',
    col('gross_amount') * col('discount_pct') / 100
).withColumn(
    'net_amount',
    col('gross_amount') - col('discount_amount')
)
cleaned_booking_df.show()


+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|discount_amount|net_amount|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|           50.0|     450.0|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|         900|            0.0|     900.0|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      1|             220|           5|     Cancelled|           UPI|   

In [0]:
#12
cleaned_booking_df = cleaned_booking_df.withColumn(
    'booking_size',
    when(col('tickets') >= 4, 'Large')
    .otherwise('Regular')
)
cleaned_booking_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|discount_amount|net_amount|booking_size|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|           50.0|     450.0|     Regular|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|         900|            0.0|     900.0|     Regular|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      

In [0]:
#13
cleaned_booking_df.sort(
    col('net_amount').desc()
).show(5)

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|discount_amount|net_amount|booking_size|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+
|     B1023|     C005|  River of Dreams|  2026-10-12|2026-10-14|      5|             320|          10|     Confirmed|          Card|        1600|          160.0|    1440.0|       Large|
|     B1029|     C005|Starlight Express|  2026-10-15|2026-10-17|      4|             320|           5|     Confirmed|          Card|        1280|           64.0|    1216.0|       Large|
|     B1013|     C001|  The Last Island|  2026-10-07|2026-10-09|      

C. Aggregations and joins

In [0]:
#14
from pyspark.sql.functions import count
cleaned_booking_df.groupBy('booking_status').agg(
    count('*').alias('total')
).show()

+--------------+-----+
|booking_status|total|
+--------------+-----+
|      Refunded|    3|
|     Confirmed|   25|
|     Cancelled|    4|
+--------------+-----+



In [0]:
#15
confirmed_bookings = cleaned_booking_df.filter(
    col('booking_status') == 'Confirmed'
)
confirmed_bookings.groupBy('movie_title').agg(
    sum('tickets').alias('total_tickets'),
    sum('net_amount').alias('total_revenue')
).show()

+-----------------+-------------+-------------+
|      movie_title|total_tickets|total_revenue|
+-----------------+-------------+-------------+
|  River of Dreams|           10|       2380.0|
|  The Last Island|           26|       5905.0|
|         Code Red|           13|       3087.0|
|Starlight Express|           23|       5500.0|
+-----------------+-------------+-------------+



In [0]:
#16
confirmed_bookings.groupBy('cinema_id').agg(
    sum("net_amount").alias('total_revenue')
).sort(
    col('total_revenue').desc()
).show(3)

+---------+-------------+
|cinema_id|total_revenue|
+---------+-------------+
|     C005|       3840.0|
|     C001|       3075.0|
|     C002|       2610.0|
+---------+-------------+
only showing top 3 rows


In [0]:
#17
left_join = cleaned_booking_df.join(
    cinemas_df,
    on = 'cinema_id',
    how = 'left'
)
display(left_join)

cinema_id,booking_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,discount_amount,net_amount,booking_size,cinema_name,facilities,location,screen_count
C001,B1001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,50.0,450.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,B1002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900,0.0,900.0,Regular,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,B1003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220,11.0,209.0,Regular,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,B1004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720,0.0,720.0,Large,Coastal Cinema,List(Parking),"List(Chennai, South)",3
C005,B1005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640,96.0,544.0,Regular,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,B1006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,0.0,1050.0,Large,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4
C001,B1007,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,0.0,750.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,B1008,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600,60.0,540.0,Regular,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,B1009,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880,44.0,836.0,Large,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,B1010,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180,0.0,180.0,Regular,Coastal Cinema,List(Parking),"List(Chennai, South)",3


In [0]:
#18
left_join.select(
    'cinema_name',
    'location',
    'movie_title',
    'booking_status'
).show()

+--------------------+------------------+-----------------+--------------+
|         cinema_name|          location|      movie_title|booking_status|
+--------------------+------------------+-----------------+--------------+
|      Galaxy Screens| {Hyderabad, West}|Starlight Express|     Confirmed|
|Silverlight Multi...|{Bengaluru, North}|  The Last Island|     Confirmed|
|        Metro Movies| {Mumbai, Central}|         Code Red|     Cancelled|
|      Coastal Cinema|  {Chennai, South}|  River of Dreams|     Confirmed|
|    Capital Cineplex|     {Delhi, East}|Starlight Express|     Confirmed|
|      Orchid Screens|      {Pune, West}|  The Last Island|     Confirmed|
|      Galaxy Screens| {Hyderabad, West}|         Code Red|     Confirmed|
|Silverlight Multi...|{Bengaluru, North}|  River of Dreams|      Refunded|
|        Metro Movies| {Mumbai, Central}|Starlight Express|     Confirmed|
|      Coastal Cinema|  {Chennai, South}|  The Last Island|     Confirmed|
|    Capital Cineplex|   

In [0]:
#19
confirmed_bookings = left_join.filter(
    col('booking_status') == 'Confirmed'
)
confirmed_bookings.groupBy('location.city').agg(
    sum('net_amount').alias('total_revenue')
).show()
#identify no matching cinema master
left_join.filter(
    col('cinema_name').isNull()
).select('booking_id').show()


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Delhi|       3840.0|
|  Chennai|       2097.0|
|Hyderabad|       3075.0|
|     Pune|       2520.0|
|     NULL|        420.0|
|   Mumbai|       2310.0|
|Bengaluru|       2610.0|
+---------+-------------+

+----------+
|booking_id|
+----------+
|     B1030|
+----------+



D. JSON, dates and DataFrame quality

In [0]:
#20
left_join.select(
    'cinema_id',
    'cinema_name',
    'location.zone'
).show()

+---------+--------------------+-------+
|cinema_id|         cinema_name|   zone|
+---------+--------------------+-------+
|     C001|      Galaxy Screens|   West|
|     C002|Silverlight Multi...|  North|
|     C003|        Metro Movies|Central|
|     C004|      Coastal Cinema|  South|
|     C005|    Capital Cineplex|   East|
|     C006|      Orchid Screens|   West|
|     C001|      Galaxy Screens|   West|
|     C002|Silverlight Multi...|  North|
|     C003|        Metro Movies|Central|
|     C004|      Coastal Cinema|  South|
|     C005|    Capital Cineplex|   East|
|     C006|      Orchid Screens|   West|
|     C001|      Galaxy Screens|   West|
|     C002|Silverlight Multi...|  North|
|     C003|        Metro Movies|Central|
|     C004|      Coastal Cinema|  South|
|     C005|    Capital Cineplex|   East|
|     C006|      Orchid Screens|   West|
|     C001|      Galaxy Screens|   West|
|     C002|Silverlight Multi...|  North|
+---------+--------------------+-------+
only showing top

In [0]:
#21
from pyspark.sql.functions import explode
left_join.select(
    'cinema_id',
    'cinema_name',
    explode('facilities').alias('facility')
).show()

+---------+--------------------+-----------------+
|cinema_id|         cinema_name|         facility|
+---------+--------------------+-----------------+
|     C001|      Galaxy Screens|          Parking|
|     C001|      Galaxy Screens|       Food Court|
|     C001|      Galaxy Screens|Wheelchair Access|
|     C002|Silverlight Multi...|          Parking|
|     C002|Silverlight Multi...|       Food Court|
|     C003|        Metro Movies|       Food Court|
|     C003|        Metro Movies|Wheelchair Access|
|     C004|      Coastal Cinema|          Parking|
|     C005|    Capital Cineplex|          Parking|
|     C005|    Capital Cineplex|       Food Court|
|     C005|    Capital Cineplex|Wheelchair Access|
|     C006|      Orchid Screens|       Food Court|
|     C006|      Orchid Screens|Wheelchair Access|
|     C001|      Galaxy Screens|          Parking|
|     C001|      Galaxy Screens|       Food Court|
|     C001|      Galaxy Screens|Wheelchair Access|
|     C002|Silverlight Multi...

In [0]:
#22
from pyspark.sql.functions import to_date, date_diff
left_join = left_join.withColumn(
    'booking_date',
    to_date(col('booking_date'))
).withColumn(
    'show_date',
    to_date(col('show_date'))
).withColumn(
    'days_between_booking_and_show',
    date_diff('show_date','booking_date')
)
left_join.show()

+---------+----------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+--------------------+--------------------+------------------+------------+-----------------------------+
|cinema_id|booking_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|discount_amount|net_amount|booking_size|         cinema_name|          facilities|          location|screen_count|days_between_booking_and_show|
+---------+----------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+---------------+----------+------------+--------------------+--------------------+------------------+------------+-----------------------------+
|     C001|     B1001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UP

In [0]:
#23 
left_join.groupBy('booking_id').agg(
    count('*').alias('count')
).filter(
    col('count') > 1
).select('booking_id').show()
left_join = left_join.dropDuplicates(['booking_id'])

+----------+
|booking_id|
+----------+
+----------+



E. Window functions and Delta

In [0]:
#24
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
confirmed_bookings = cleaned_booking_df.filter(
    col('booking_status') == 'Confirmed'
)
group = confirmed_bookings.groupBy('movie_title').agg(
    sum('net_amount').alias('total_revenue')
)
window_spec = Window.orderBy(col('total_revenue').desc())
movie_rating = group.withColumn(
    'rank',
    rank().over(window_spec)
)
display(movie_rating)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


movie_title,total_revenue,rank
The Last Island,5905.0,1
Starlight Express,5500.0,2
Code Red,3087.0,3
River of Dreams,2380.0,4
